# Mais um concorrente baixa o preço? A concorrência nos concursos públicos portugueses

**Pergunta.** Nos concursos públicos e nas consultas prévias, quanto mais baixo é o preço contratual, face ao preço base do
procedimento, quando concorrem mais empresas? E com que frequência um concurso recebe uma só proposta?

**Dados.** 1,55 milhões de contratos públicos publicados no Portal BASE (IMPIC), de janeiro de 2019 a 24 de outubro de 2026,
os mesmos ficheiros em bruto que o [projeto dbt](../models) usa. Este notebook trabalha a partir dos ficheiros Parquet em bruto em pandas,
de forma independente dos modelos SQL, e confirma que ambos dão os mesmos totais.

**Resposta curta** (os números são calculados mais abaixo):

* Com uma só proposta, os concursos públicos fecham numa mediana de **99,3%** do preço base. Com dez ou mais propostas, a **79,4%**.
* Controlando a categoria, o ano, o tipo de contrato e a dimensão do contrato, **cada duplicação do número de concorrentes
  corresponde a um preço cerca de 5,7% mais baixo** (concursos públicos e consultas prévias em conjunto).
  O resultado mantém-se dentro da mesma entidade adjudicante e dentro da mesma classe CPV e escalão de dimensão: −5,5% a −5,8%
  ([secção 6](#robustness)), de cerca de −10% em arquitetura e engenharia a −3% em energia.
* Os concursos com uma só proposta são cada vez mais comuns: **9,0% dos concursos públicos em 2019, 12,6% em 2025**; nas consultas
  prévias, **de 35,6% para 47,0%**.

Esta é uma análise descritiva de registos publicados. Mostra associações; não diz que alguma entidade adjudicante ou
fornecedor tenha agido mal, nem prova que mais concorrentes *causem* preços mais baixos (ver a última secção).

**Os gráficos são interativos** (passe o cursor para ver os valores; clique numa legenda ou numa barra para filtrar os outros gráficos).
São feitos com [Altair](https://altair-viz.github.io/), que escreve especificações Vega-Lite executadas no browser.
O GitHub mostra os notebooks sem JavaScript, por isso abra a
[versão interativa](https://guilhermerigonato.com/public-procurement-pt/notebook/pt/) para os usar.
O código e os nomes das colunas ficam em inglês, como no repositório.

**Para correr:** `pip install pandas pyarrow altair statsmodels pyfixest`, ponha os ficheiros do Portal BASE em `data/raw/`
(ver [`scripts/get_data.py`](../scripts/get_data.py)) e execute todas as células (cerca de 2 minutos).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import altair as alt
import statsmodels.formula.api as smf

RAW = Path("../data/raw")
SEEDS = Path("../seeds")
pd.set_option("display.float_format", "{:,.3f}".format)

# Tema sóbrio dos gráficos, igual em todo o portfólio
BLUE, ORANGE, GREY, MUTED, GRID = "#2a78d6", "#eb6834", "#c9c8c3", "#52514e", "#e4e3df"

@alt.theme.register("portfolio", enable=True)
def portfolio_theme():
    axis = {"gridColor": GRID, "domainColor": MUTED, "tickColor": MUTED, "labelColor": MUTED,
            "titleColor": MUTED, "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": "normal"}
    return {"config": {"view": {"stroke": None}, "axis": axis, "legend": {"labelColor": MUTED, "titleColor": MUTED},
                       "title": {"anchor": "start", "fontSize": 13, "color": "#0b0b0b"},
                       "font": "system-ui, -apple-system, Segoe UI, sans-serif"}}

## 1. Carregar e limpar os ficheiros em bruto

Todas as colunas da fonte são texto. A limpeza segue as mesmas regras do modelo de staging do dbt
[`stg_contracts.sql`](../models/staging/stg_contracts.sql):

* os textos vazios e o marcador `∅` passam a valores em falta;
* 4566 identificadores de contrato aparecem duas vezes no mesmo ficheiro anual; fica uma linha por identificador (o ficheiro mais recente primeiro);
* os valores usam `.` como separador decimal; os preços negativos são tratados como em falta;
* o número de concorrentes é o número de linhas em `concorrentes` (só é indicado em alguns procedimentos).

Os nomes dos fornecedores e as descrições dos contratos não são usados em nenhum ponto deste notebook.

In [2]:
COLS = ["idcontrato", "idprocedimento", "tipoContrato", "tipoprocedimento", "concorrentes",
        "precoContratual", "precoBaseProcedimento", "CPV", "TipoCriterioAdjudicacao", "nAnuncio",
        "adjudicante", "prazoExecucao"]

frames = []
for f in sorted(RAW.glob("contratos_*.parquet")):
    part = pd.read_parquet(f, columns=COLS)
    part["file_year"] = int(f.stem.split("_")[1])
    frames.append(part)
raw = pd.concat(frames, ignore_index=True)

for c in COLS:
    s = raw[c].str.strip()
    raw[c] = s.mask(s.isin(["", "∅"]))

print(f"{len(raw):,} linhas em {len(frames)} ficheiros anuais")

1 555 252 linhas em 8 ficheiros anuais


In [3]:
# Uma linha por contrato: o ficheiro mais recente primeiro, depois as linhas que indicam o critério de adjudicação
df = (raw.sort_values(["idcontrato", "file_year", "TipoCriterioAdjudicacao", "nAnuncio"],
                      ascending=[True, False, True, True], na_position="last")
         .drop_duplicates("idcontrato")
         .reset_index(drop=True))

def amount(s):
    x = pd.to_numeric(s, errors="coerce").round(2)
    return x.mask(x < 0)

df["value"] = amount(df["precoContratual"])
df["base_price"] = amount(df["precoBaseProcedimento"])
df["bidders"] = df["concorrentes"].str.count("\n").add(1)          # em falta quando não é indicado
df["cpv_division"] = df["CPV"].str.extract(r"^(\d{2})\d{6}")[0].fillna("00")
df["contract_type"] = df["tipoContrato"].fillna("Not reported").str.split("\n").str[0]
df["year"] = df["file_year"]

procedures = pd.read_csv(SEEDS / "procedure_types.csv")
cpv = pd.read_csv(SEEDS / "cpv_divisions.csv", dtype={"cpv_division": str})
df = (df.assign(procedure_name_pt=df["tipoprocedimento"].fillna("Not reported"))
        .merge(procedures[["procedure_name_pt", "procedure_group"]], on="procedure_name_pt", how="left")
        .merge(cpv[["cpv_division", "cpv_short_name"]], on="cpv_division", how="left"))

print(f"{len(df):,} contratos, €{df['value'].sum() / 1e9:,.2f}bn sem IVA")

1 550 674 contratos, 120,54 mil M€ sem IVA


**Conciliação com o projeto dbt.** Os modelos SQL indicam 1 550 674 contratos e 120,54 mil M€. Se a limpeza em pandas se afastar das regras do SQL, esta célula falha.

In [4]:
assert len(df) == 1_550_674, len(df)
assert round(df["value"].sum() / 1e9, 2) == 120.54
assert df["procedure_group"].notna().all(), "procedure missing from the seed"
print("o pandas e o dbt coincidem")

o pandas e o dbt coincidem


In [5]:
(df.groupby("procedure_group")
   .agg(contracts=("idcontrato", "size"), value_bn=("value", lambda v: v.sum() / 1e9),
        bidders_reported=("bidders", lambda b: b.notna().mean()))
   .assign(share_of_value=lambda t: t["value_bn"] / t["value_bn"].sum())
   .sort_values("value_bn", ascending=False))

,contracts,value_bn,bidders_reported,share_of_value
procedure_group,,,,
Concurso público,212366,61.106,0.795,0.507
Direct award,729716,19.794,0.192,0.164
Framework call-off,215513,18.270,0.428,0.152
Consulta prévia,305655,10.189,0.795,0.085
Restricted or negotiated,3301,7.946,0.597,0.066
Excluded or exempt,42204,3.045,0.999,0.025
Simplified direct award,41905,0.189,0.000,0.002
Other,14,0.005,0.000,0.000


O número de concorrentes vem indicado em quase todos os concursos públicos e consultas prévias, e quase nunca nos
ajustes diretos (que, por definição, convidam uma só empresa). Por isso, o resto do notebook olha para os dois grupos de
procedimentos concorrenciais onde a contagem existe: **concursos públicos** e **consultas prévias**.

## 2. Com que frequência um concurso recebe uma só proposta?

In [6]:
COMPETITIVE = ["Open tender", "Prior consultation"]
comp = df[df["procedure_group"].isin(COMPETITIVE) & (df["bidders"] > 0)].copy()

single_by_year = (comp.assign(single=comp["bidders"].eq(1))
                      .pivot_table(index="year", columns="procedure_group", values="single", aggfunc="mean"))
single_by_year.style.format("{:.1%}")

procedure_group,Concurso público,Consulta prévia
year,,
2019,9.0%,35.6%
2020,8.2%,36.7%
2021,8.4%,42.1%
2022,9.7%,46.1%
2023,9.4%,42.8%
2024,10.1%,43.8%
2025,12.6%,47.0%
2026,13.1%,47.5%


In [7]:
PROC_COLOURS = alt.Scale(domain=COMPETITIVE, range=[BLUE, ORANGE])
pick_proc = alt.selection_point(fields=["procedure_group"], bind="legend")   # clique na legenda para isolar

long = single_by_year.reset_index().melt("year", var_name="procedure_group", value_name="single_bid_share")
alt.Chart(long, title="Procedimentos com uma só proposta (2026 = janeiro a 24 de outubro)").mark_line(
    point=alt.OverlayMarkDef(size=60), strokeWidth=2
).encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("single_bid_share:Q", title=None, axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 0.55])),
    color=alt.Color("procedure_group:N", scale=PROC_COLOURS, title="Clique para isolar",
                    legend=alt.Legend(orient="top")),
    opacity=alt.condition(pick_proc, alt.value(1), alt.value(0.15)),
    tooltip=[alt.Tooltip("year:O", title="Ano"), alt.Tooltip("procedure_group:N", title="Procedimento"),
             alt.Tooltip("single_bid_share:Q", title="Uma só proposta", format=".1%")],
).add_params(pick_proc).properties(width=620, height=260)

alt.Chart(...)

In [8]:
# Que categorias recebem menos propostas? Só concursos públicos, categorias com pelo menos 1000 contratos
ot = comp[comp["procedure_group"] == "Open tender"]
by_cpv = (ot.groupby("cpv_short_name")
            .agg(open_tenders=("idcontrato", "size"), single_bid_share=("bidders", lambda b: b.eq(1).mean()),
                 median_bidders=("bidders", "median"))
            .query("open_tenders >= 1000")
            .sort_values("single_bid_share", ascending=False))
by_cpv.head(8).style.format({"single_bid_share": "{:.1%}", "open_tenders": "{:,}", "median_bidders": "{:.0f}"})

,open_tenders,single_bid_share,median_bidders
cpv_short_name,,,
Software,"3,214",36.8%,2
Serviços de TI,"5,016",32.3%,2
Outros serviços comunitários,"1,280",26.4%,3
Reparação e manutenção,"5,286",20.2%,3
Serviços postais e de telecomunicações,"1,446",19.4%,2
"Lazer, cultura e desporto","1,387",17.8%,4
Máquinas industriais,"1,791",16.0%,4
Equipamento de telecomunicações,"2,084",15.7%,4


## 3. Preço contratual face ao preço base, por número de concorrentes

O preço base é o máximo que a entidade adjudicante está disposta a pagar, fixado antes de chegarem as propostas. O rácio
*preço contratual / preço base* é a parte desse teto que a entidade acaba por pagar: 1,00 significa que pagou o
teto todo, 0,80 significa um desconto de 20%.

As mesmas regras do mart do dbt [`mart_price_vs_base`](../models/marts/mart_price_vs_base.sql): só procedimentos com um
único contrato (um procedimento com vários lotes publica um só preço base para todos os lotes), ambos os preços acima de zero e
rácios entre 0,2 e 5 (fora desse intervalo, o preço base está quase sempre mal escrito).

In [9]:
contracts_per_procedure = df.groupby("idprocedimento")["idcontrato"].transform("size")
df["price_ratio"] = np.where((contracts_per_procedure == 1) & (df["base_price"] > 0) & (df["value"] > 0),
                             df["value"] / df["base_price"], np.nan)

sample = df[df["procedure_group"].isin(COMPETITIVE) & (df["bidders"] > 0)
            & df["price_ratio"].between(0.2, 5)].copy()
sample["bidders_capped"] = sample["bidders"].clip(upper=10).astype(int)

ratio_table = sample.pivot_table(index="bidders_capped", columns="procedure_group",
                                 values="price_ratio", aggfunc="median")
counts = sample.pivot_table(index="bidders_capped", columns="procedure_group",
                            values="price_ratio", aggfunc="size")
print(f"{len(sample):,} contratos na amostra")
ratio_table.join(counts, rsuffix=" (contracts)").rename(index={10: "10+"})

272 342 contratos na amostra


procedure_group,Concurso público,Consulta prévia,Concurso público (contratos),Consulta prévia (contratos)
bidders_capped,,,,
1,0.993,0.999,14843,100183
2,0.962,0.969,14017,48316
3,0.931,0.974,12116,35931
4,0.919,0.891,9692,5891
5,0.895,0.886,7237,2060
6,0.884,0.853,5415,599
7,0.867,0.855,3964,264
8,0.856,0.840,3012,127
9,0.844,0.820,2246,60


### Explorar por categoria: três gráficos ligados

Clique numa categoria à esquerda (duplo clique para limpar): os dois gráficos da direita redesenham-se para essa categoria.
Clique num procedimento na legenda para esbater o outro nos dois gráficos. Passe o cursor por qualquer marca para ver os valores.
As medianas são pré-calculadas em pandas para cada categoria, por isso a página só filtra tabelas pequenas
(uma combinação de categoria e número de propostas com menos de 30 contratos fica de fora).

In [10]:
def by_category(frame, keys, agg):
    # A mesma agregação por categoria CPV e para todas as categorias juntas
    per_cat = (frame.groupby(["cpv_short_name", *keys]).agg(**agg).reset_index()
                    .rename(columns={"cpv_short_name": "category"}))
    total = frame.groupby(keys).agg(**agg).reset_index().assign(category="All categories")
    return pd.concat([total, per_cat], ignore_index=True)

price_cat = by_category(sample, ["procedure_group", "bidders_capped"],
                        dict(median_ratio=("price_ratio", "median"), contracts=("price_ratio", "size")))
price_cat = price_cat[price_cat["contracts"] >= 30]
year_cat = by_category(comp, ["procedure_group", "year"],
                       dict(single_bid_share=("bidders", lambda b: b.eq(1).mean()), procedures=("bidders", "size")))
cat_bars = by_cpv.reset_index().rename(columns={"cpv_short_name": "category"})

click_cat = alt.selection_point(name="click_cat", fields=["category"], on="click", clear="dblclick")
proc = alt.selection_point(fields=["procedure_group"], bind="legend")
chosen = "(click_cat.category && length(click_cat.category)) ? click_cat.category[0] : 'All categories'"
fade = alt.condition(proc, alt.value(1), alt.value(0.15))

left = alt.Chart(cat_bars, title="Concursos públicos com uma só proposta · clique numa categoria").mark_bar(
    cornerRadiusEnd=3, height=13
).encode(
    y=alt.Y("category:N", sort="-x", title=None, axis=alt.Axis(labelLimit=210)),
    x=alt.X("single_bid_share:Q", title=None, axis=alt.Axis(format="%")),
    color=alt.condition(click_cat, alt.value(BLUE), alt.value(GREY)),
    tooltip=[alt.Tooltip("category:N", title="Categoria"),
             alt.Tooltip("single_bid_share:Q", title="Uma só proposta", format=".1%"),
             alt.Tooltip("open_tenders:Q", title="Concursos públicos", format=","),
             alt.Tooltip("median_bidders:Q", title="Mediana de propostas")],
).add_params(click_cat).properties(width=260, height=alt.Step(17))

years = alt.Chart(year_cat).transform_filter(f"datum.category == ({chosen})").mark_line(
    point=True, strokeWidth=2
).encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("single_bid_share:Q", title="Com uma só proposta", axis=alt.Axis(format="%")),
    color=alt.Color("procedure_group:N", scale=PROC_COLOURS, title=None, legend=alt.Legend(orient="top")),
    opacity=fade,
    tooltip=[alt.Tooltip("year:O", title="Ano"), alt.Tooltip("procedure_group:N", title="Procedimento"),
             alt.Tooltip("single_bid_share:Q", title="Uma só proposta", format=".1%"),
             alt.Tooltip("procedures:Q", format=",")],
).add_params(proc).properties(width=400, height=190,
                              title=alt.Title(alt.expr(f"'Com uma só proposta, por ano · ' + ({chosen})")))

prices = alt.Chart(price_cat).transform_filter(f"datum.category == ({chosen})").mark_bar().encode(
    x=alt.X("bidders_capped:O", title="Número de propostas (10 = dez ou mais)", axis=alt.Axis(labelAngle=0)),
    xOffset="procedure_group:N",
    y=alt.Y("median_ratio:Q", title="Preço mediano / preço base", axis=alt.Axis(format="%"),
            scale=alt.Scale(domain=[0.6, 1.05], clamp=True)),
    color=alt.Color("procedure_group:N", scale=PROC_COLOURS, legend=None),
    opacity=fade,
    tooltip=[alt.Tooltip("bidders_capped:O", title="Propostas"), alt.Tooltip("procedure_group:N", title="Procedimento"),
             alt.Tooltip("median_ratio:Q", title="Preço mediano / base", format=".1%"),
             alt.Tooltip("contracts:Q", format=",")],
).properties(width=400, height=190,
             title=alt.Title(alt.expr(f"'Preço pago por número de propostas · ' + ({chosen})")))

alt.hconcat(left, alt.vconcat(years, prices), spacing=30)

alt.HConcatChart(...)

A descida é constante: cada proposta a mais baixa o preço mediano. Mas a comparação acima mistura contratos muito
diferentes. As obras rodoviárias recebem mais concorrentes do que o software hospitalar, e os contratos grandes atraem mais concorrentes do que
os pequenos. A regressão abaixo mantém essas diferenças constantes.

## 4. Regressão: o efeito no preço de haver mais concorrentes, tudo o resto igual

$$\log\left(\frac{\text{preço contratual}}{\text{preço base}}\right) = \beta \cdot \log(\text{concorrentes}) + \gamma \cdot \log(\text{preço base})
+ \text{procedimento} + \text{divisão CPV} + \text{ano} + \text{tipo de contrato} + \varepsilon$$

Com os dois lados em logaritmos, $2^{\beta} - 1$ é a variação do preço quando o número de concorrentes duplica.
Os erros-padrão são robustos à heterocedasticidade (HC1).

In [11]:
sample["log_ratio"] = np.log(sample["price_ratio"])
sample["log_bidders"] = np.log(sample["bidders"])
sample["log_base"] = np.log(sample["base_price"])

FE = "C(procedure_group) + C(cpv_division) + C(year) + C(contract_type)"
model = smf.ols(f"log_ratio ~ log_bidders + log_base + {FE}", data=sample).fit(cov_type="HC1")

beta, se = model.params["log_bidders"], model.bse["log_bidders"]
print(f"beta = {beta:.4f} (s.e. {se:.4f}), n = {int(model.nobs):,}, R² = {model.rsquared:.3f}")
print(f"Duplicar o número de concorrentes: {2 ** beta - 1:+.1%} no preço "
      f"(95% CI {2 ** (beta - 1.96 * se) - 1:+.1%} to {2 ** (beta + 1.96 * se) - 1:+.1%})")

beta = −0,0842 (e.p. 0,0007), n = 272 342, R² = 0,132
Duplicar o número de concorrentes: −5,7% no preço (IC 95% −5,8% a −5,6%)


In [12]:
# O mesmo modelo com uma dummy por número de propostas (1 proposta é a referência), para ver a forma sem assumir uma curva logarítmica
sample["bids"] = sample["bidders"].clip(upper=6).astype(int).astype(str).replace("6", "6+")
dummies = smf.ols(f"log_ratio ~ C(bids) + log_base + {FE}", data=sample).fit(cov_type="HC1")

effects = pd.DataFrame({
    "price_vs_single_bid": np.exp(dummies.params.filter(like="C(bids)")) - 1,
    "ci_low": np.exp(dummies.conf_int().filter(like="C(bids)", axis=0)[0]) - 1,
    "ci_high": np.exp(dummies.conf_int().filter(like="C(bids)", axis=0)[1]) - 1,
})
effects.index = [i.split("T.")[1].rstrip("]") + " bids" for i in effects.index]
effects.style.format("{:+.1%}")

,price_vs_single_bid,ci_low,ci_high
2 propostas,-5.4%,-5.5%,-5.2%
3 propostas,-6.5%,-6.7%,-6.4%
4 propostas,-11.2%,-11.6%,-10.9%
5 propostas,-12.4%,-12.8%,-12.0%
6+ propostas,-17.5%,-17.9%,-17.2%


In [13]:
eff = effects.reset_index(names="bids")
base = alt.Chart(eff).encode(y=alt.Y("bids:N", sort=None, title=None))
alt.layer(
    alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color=MUTED).encode(x="x:Q"),
    base.mark_rule(strokeWidth=3, color=BLUE).encode(x="ci_low:Q", x2="ci_high:Q"),
    base.mark_point(filled=True, size=140, color=BLUE, opacity=1).encode(
        x=alt.X("price_vs_single_bid:Q", title="Variação do preço contratual / preço base (intervalo de confiança de 95%)",
                axis=alt.Axis(format="%"), scale=alt.Scale(domain=[-0.22, 0.02])),
        tooltip=[alt.Tooltip("bids:N", title="Propostas"),
                 alt.Tooltip("price_vs_single_bid:Q", title="Efeito", format="+.1%"),
                 alt.Tooltip("ci_low:Q", title="IC 95% mínimo", format="+.1%"),
                 alt.Tooltip("ci_high:Q", title="IC 95% máximo", format="+.1%")]),
    base.mark_text(dy=-14, fontSize=11).encode(x="price_vs_single_bid:Q",
                                               text=alt.Text("price_vs_single_bid:Q", format="+.1%")),
).properties(width=620, height=240, title="Preço face a um concurso com uma só proposta, tudo o resto igual")

alt.LayerChart(...)

## 5. Quanto vale, em ordem de grandeza

Uma ilustração, não uma estimativa de poupança: se os concursos públicos com uma só proposta tivessem preços como concursos comparáveis com
duas propostas, quanto mais baixo seria o seu valor contratual?

In [14]:
two_bids = np.exp(dummies.params["C(bids)[T.2]"]) - 1
single_ot = sample[(sample["procedure_group"] == "Open tender") & (sample["bidders"] == 1)]
print(f"Concursos públicos com uma só proposta na amostra: {len(single_ot):,} contratos, "
      f"€{single_ot['value'].sum() / 1e9:,.2f}bn contratados")
print(f"Ao preço de duas propostas ({two_bids:+.1%}): cerca de €{-two_bids * single_ot['value'].sum() / 1e6:,.0f}m a menos")

Concursos públicos com uma só proposta na amostra: 14 843 contratos, 5,80 mil M€ contratados
Ao preço de duas propostas (−5,4%): cerca de 312 M€ a menos


<a id="robustness"></a>
## 6. Robustez: a mesma entidade, a mesma categoria, a mesma dimensão

Uma objeção justa à secção 4: os concursos que recebem uma só proposta podem ser diferentes à partida, por exemplo um caderno de
encargos muito específico ou um prazo curto, e essas características podem, por si só, subir o preço. Duas verificações
estreitam a comparação:

* **Efeitos fixos da entidade adjudicante.** Cada entidade só é comparada consigo própria: o *mesmo* hospital ou
  município paga menos nos concursos que tiveram mais propostas? Isto absorve a forma como cada entidade escreve o caderno de encargos e
  fixa o preço base.
* **Células categoria × dimensão.** Cada concurso só é comparado com concursos da mesma classe CPV (5 dígitos, por exemplo obras
  rodoviárias e não toda a construção) e do mesmo decil de preço base. Uma variante usa o código CPV completo de 8 dígitos.

Os erros-padrão são agrupados por entidade adjudicante. Os efeitos fixos são absorvidos com
[pyfixest](https://github.com/py-econometrics/pyfixest), que lida rapidamente com milhares de entidades.

In [15]:
import pyfixest as pf
import warnings

s = sample.copy()
s["buyer"] = s["adjudicante"].str.extract(r"^(\d{9})")[0].fillna("not reported")   # só o NIF, nunca o nome
s["cpv5"] = s["CPV"].str.extract(r"^(\d{5})\d{3}")[0].fillna("00000")
s["cpv8"] = s["CPV"].str.extract(r"^(\d{8})")[0].fillna("00000000")
s["price_band"] = pd.qcut(s["log_base"], 10, labels=False)                    # decis do preço base
s["cell"] = s["cpv5"] + "_" + s["price_band"].astype(str)
s["buyer_cpv"] = s["buyer"] + "_" + s["cpv_division"]
duration = pd.to_numeric(s["prazoExecucao"], errors="coerce").clip(lower=0)
s["duration_band"] = pd.qcut(duration, 5, labels=False, duplicates="drop").fillna(-1)

SPECS = {
    "Section 4 model (CPV division, year, size)": "procedure_group + cpv_division + year + contract_type",
    "+ contracting-authority fixed effects": "procedure_group + cpv_division + year + contract_type + buyer",
    "CPV class x base-price decile": "procedure_group + year + contract_type + cell",
    "CPV class x decile + buyer": "procedure_group + year + contract_type + cell + buyer",
    "+ buyer x CPV division + contract duration": "procedure_group + year + contract_type + cell + buyer_cpv + duration_band",
    "8-digit CPV x decile + buyer": "procedure_group + year + contract_type + cpv8^price_band + buyer",
}

def doubling(fit, var="log_bidders"):
    b, se = fit.coef()[var], fit.se()[var]
    return 2 ** b - 1, 2 ** (b - 1.96 * se) - 1, 2 ** (b + 1.96 * se) - 1

rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")      # o pyfixest avisa das células com um só caso que descarta
    for name, fe in SPECS.items():
        fit = pf.feols(f"log_ratio ~ log_bidders + log_base | {fe}", data=s, vcov={"CRV1": "buyer"})
        rows.append((name, fit._N, *doubling(fit)))
robust = pd.DataFrame(rows, columns=["model", "contracts", "doubling_bidders", "ci_low", "ci_high"]).set_index("model")

varied = s.groupby("buyer")["bidders"].nunique().gt(1)
print(f"{s['buyer'].nunique():,} entidades adjudicantes; {s['buyer'].map(varied).mean():.0%} da amostra vem de entidades "
      "cujos concursos tiveram números de propostas diferentes")
robust.style.format({"contracts": "{:,}", "doubling_bidders": "{:+.1%}", "ci_low": "{:+.1%}", "ci_high": "{:+.1%}"})

6804 entidades adjudicantes; 98% da amostra vem de entidades cujos concursos tiveram números de propostas diferentes


,contracts,doubling_bidders,ci_low,ci_high
model,,,,
"Section 4 model (CPV division, year, size)","272,342",-5.7%,-5.9%,-5.4%
+ contracting-authority fixed effects,"270,434",-5.8%,-6.0%,-5.6%
CPV class x base-price decile,"268,153",-5.5%,-5.7%,-5.3%
CPV class x decile + buyer,"266,241",-5.7%,-5.8%,-5.5%
+ buyer x CPV division + contract duration,"253,782",-5.5%,-5.7%,-5.4%
8-digit CPV x decile + buyer,"261,545",-5.7%,-5.9%,-5.5%


O efeito quase não se mexe: cerca de **−5,5% a −5,8% por duplicação** em todas as especificações, contra −5,7% na secção 4.
A mesma entidade, a comprar o mesmo tipo de coisa com uma dimensão parecida, continua a pagar menos quando concorrem mais empresas.

### É igual em todas as categorias?

Os −5,7% são uma média. A célula abaixo ajusta o modelo mais exigente (classe CPV × decil + entidade adjudicante) em separado para cada
divisão CPV com pelo menos 2000 contratos.

In [16]:
cpv_names = cpv.set_index("cpv_division")["cpv_short_name"]
rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for division, g in s.groupby("cpv_division"):
        if len(g) < 2000:
            continue
        fit = pf.feols("log_ratio ~ log_bidders + log_base | procedure_group + year + contract_type + cell + buyer",
                       data=g, vcov={"CRV1": "buyer"})
        rows.append((cpv_names.get(division, division), fit._N, *doubling(fit), g["bidders"].eq(1).mean()))
by_division = (pd.DataFrame(rows, columns=["category", "contracts", "doubling_bidders", "ci_low", "ci_high", "single_bid_share"])
                 .sort_values("doubling_bidders").reset_index(drop=True))
negative = (by_division["ci_high"] < 0).sum()
print(f"{len(by_division)} categorias, {by_division['contracts'].sum():,} contratos; "
      f"claramente negativo (todo o IC 95% abaixo de zero) em {negative}")

base = alt.Chart(by_division).encode(y=alt.Y("category:N", sort=None, title=None, axis=alt.Axis(labelLimit=260)))
alt.layer(
    alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(color=MUTED).encode(x="x:Q"),
    base.mark_rule(strokeWidth=3, color=BLUE, opacity=0.35).encode(x="ci_low:Q", x2="ci_high:Q"),
    base.mark_point(filled=True, size=70, color=BLUE, opacity=1).encode(
        x=alt.X("doubling_bidders:Q", title="Variação do preço quando o número de concorrentes duplica (IC 95%)",
                axis=alt.Axis(format="%")),
        tooltip=[alt.Tooltip("category:N", title="Categoria"),
                 alt.Tooltip("doubling_bidders:Q", title="Duplicar concorrentes", format="+.1%"),
                 alt.Tooltip("ci_low:Q", title="IC 95% mínimo", format="+.1%"),
                 alt.Tooltip("ci_high:Q", title="IC 95% máximo", format="+.1%"),
                 alt.Tooltip("contracts:Q", title="Contratos", format=","),
                 alt.Tooltip("single_bid_share:Q", title="Uma só proposta", format=".0%")]),
).properties(width=560, height=alt.Step(17), title="Mesma entidade, mesma classe CPV e dimensão: efeito por categoria")

28 categorias, 239 225 contratos; claramente negativo (todo o IC 95% abaixo de zero) em 27


alt.LayerChart(...)

O efeito é negativo e claro em quase todas as categorias, mas a dimensão varia. É maior nos serviços em que
o preço é sobretudo uma cotação (arquitetura e engenharia, serviços de TI) e menor onde os preços seguem um mercado
(energia e combustíveis, construção), com a hotelaria e a restauração perto de zero.

O que estas verificações ainda não conseguem manter constante: o prazo que os concorrentes tiveram para apresentar proposta e quão restritivo era
o caderno de encargos. Nenhum dos dois está no ficheiro de contratos; o conjunto de dados dos anúncios tem o prazo e é o próximo teste.

## O que isto não mostra

* **Não é causal.** Os concursos que atraem uma só proposta podem ser diferentes de formas que os dados não registam (um caderno de encargos
  muito específico, um local remoto, um prazo curto). Parte da diferença pode vir dessas diferenças, por isso o
  valor em euros acima é uma ilustração de limite superior, não dinheiro que se perdeu.
* **Os preços base são fixados pela entidade adjudicante.** Uma entidade que fixa um preço base generoso verá «descontos» maiores.
  Os controlos de CPV, ano e dimensão absorvem parte disto, não tudo.
* **Os concorrentes são os indicados.** A contagem vem do campo `concorrentes`; falta na maioria dos ajustes diretos
  e o notebook deixa-os de fora.
* **A amostra** são procedimentos de contrato único com um preço base plausível (contados na célula abaixo), por isso
  os concursos com vários lotes ficam de fora.

A secção 6 já compara cada entidade consigo própria e cada concurso com outros da mesma classe CPV e dimensão.
O passo seguinte é o conjunto de dados dos anúncios, para medir quanto tempo cada concurso esteve aberto.

In [17]:
print(f"Amostra: {len(sample):,} de {len(comp):,} concursos públicos e consultas prévias com número de propostas indicado "
      f"({len(sample) / len(comp):.0%})")

Amostra: 272 342 de 411 701 concursos públicos e consultas prévias com número de propostas indicado (66%)


## Sobre os gráficos interativos: que biblioteca para quê

Os gráficos ligados, em que uma seleção num gráfico filtra os outros, são o que faz um notebook parecer um
painel. As principais opções em Python, e onde cada uma encaixa:

| Biblioteca | Gráficos ligados | Resultado | Melhor para |
|---|---|---|---|
| **Altair** (Vega-Lite), usada aqui | Nativos: seleções e parâmetros partilhados entre gráficos | HTML estático, sem servidor | Explorações e relatórios que têm de abrir em qualquer lado, como esta página |
| **Plotly** | Cursor, zoom e legenda por gráfico; ligar gráficos exige callbacks do Dash | HTML; uma app Dash para ligar | Gráficos individuais ricos, mapas, 3D |
| **Bokeh** | Seleção ligada através de uma `ColumnDataSource` partilhada | HTML ou um servidor Bokeh para callbacks em Python | Dispersões grandes, ferramentas à medida |
| **HoloViews + Panel** | Seleções ligadas numa só chamada (`link_selections`) | Notebook ou app Panel | Transformar um notebook numa pequena app |
| **Dash / Streamlit** | Qualquer ligação, escrita como callbacks em Python | Precisa de um servidor a correr | Ferramentas internas com dados ao vivo |

O painel web deste repositório (`web/`) faz o mesmo em JavaScript simples e SVG, e as versões Power BI e Qlik
fazem-no de origem (filtro cruzado e modelo associativo).